In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import sqlite3
import requests

C:\Users\hp\AppData\Local\Temp\ipykernel_3136\796210333.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model="openai/gpt-oss-120b")

In [4]:
search_tool = DuckDuckGoSearchRun(region='us_en')

#calculator tool:_____


@tool
def calculator(a: float, b: float, operation: str) -> float:
    """Perform basic calculations."""

    if operation == "add":
        return a + b

    elif operation == "subtract":
        return a - b

    elif operation == "multiply":
        return a * b

    elif operation == "divide":
        return a / b

    return 0

In [5]:


@tool
def stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    r = requests.get(url)
    return r.json()

In [6]:
#_______binding_tools________________
tools = [search_tool, calculator, stock_price]
llm_with_tool= model.bind_tools(tools)

In [7]:
class state_state(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [8]:
def chat_node(state:state_state)->state_state:
    "llm node that may answer or request a tool call"
    message=state["messages"]
    response = llm_with_tool.invoke(message)

    return {
        'messages':[response]
    }

tool_node =ToolNode(tools)

In [9]:
graph = StateGraph(state_state)

graph.add_node('chat_node', chat_node)
graph.add_node('tool_node', tool_node)

graph.add_edge(START, 'chat_node')

graph.add_conditional_edges(
    'chat_node',
    tools_condition,
    {
        'tools': 'tool_node',
        '__end__': END
    }
)
graph.add_edge('tool_node', 'chat_node')


workflow=graph.compile()

In [10]:
out= workflow.invoke({'messages': [HumanMessage(content='what is python')]})
print('out :', out['messages'][-1].content)

out : **Python** is a high‑level, interpreted programming language known for its readability, simplicity, and versatility. It was created by **Guido van Rossum** and first released in **1991**. Here are the key points that define Python:

| Aspect | Details |
|--------|---------|
| **Paradigms** | Supports multiple programming styles: procedural, object‑oriented, functional, and imperative. |
| **Syntax** | Emphasizes clean, human‑readable code. Indentation (whitespace) is used to delimit code blocks instead of braces or keywords. |
| **Interpreted** | Code is executed by the Python interpreter (CPython is the reference implementation). No separate compilation step is required, which speeds up development and testing. |
| **Dynamic typing** | Variable types are determined at runtime, allowing rapid prototyping and flexibility. |
| **Standard Library** | Ships with a “batteries‑included” library that covers everything from file I/O, networking, and data serialization to web services, te

In [11]:
out= workflow.invoke({'messages': [HumanMessage(content='what are the stock price of iphone ')]})
print('out :', out['messages'][-1].content)

out : Apple Inc. (ticker **AAPL**) is the publicly traded company that makes the iPhone. According to the latest market data (as of 2026‑09‑15):

| Metric | Value |
|--------|-------|
| **Current price** | **$331.34** per share |
| **Open (today)** | $330.14 |
| **Day‑high** | $331.78 |
| **Day‑low** | $328.35 |
| **Previous close** | $333.08 |
| **Change** | –$1.74 (‑0.52 %) |
| **Volume** | 31,748,183 shares |

So, the iPhone itself doesn’t have a stock price, but the stock that produces it—Apple Inc.—is trading around **$331 per share** right now. If you need more detailed historical data, dividend information, or analyst outlooks, just let me know!
